![alt text](header.png)

# Notebook 4 | Post-Deployment Validation

> **Workshop: From Algorithm to Hardware: Machine Learning in Embedded Systems | April 2026**   

After converting a model to HLS and synthesizing it on FPGA, the key question is:
**does the hardware model produce the same results as the original Keras/QKeras model?**

This notebook validates using **hls4ml C-simulation**, not a numpy approximation.
C-sim uses the exact same fixed-point arithmetic that Vivado will synthesize,
and runs in seconds instead of hours.

## 1. Why validate after deployment?

Several sources of discrepancy can appear between the Keras/QKeras model and the FPGA implementation:

- **Quantization error**: fixed-point arithmetic introduces rounding
- **Numerical overflow**: if `ap_fixed<W,I>` integer range is too small, values saturate
- **Activation approximations**: hls4ml uses lookup tables for some activations
- **Accumulator growth**: summing N products requires more integer bits than the weights alone

Validation confirms the hardware model meets accuracy requirements before full synthesis.

In [ ]:
import os

# PATH Vitis HLS installation
os.environ['PATH'] = '/tools/Xilinx/Vitis_HLS/2024.1/bin:' + os.environ['PATH']
os.environ['PATH']

os.environ["XILINX_AP_INCLUDE"] = "-I/tools/Xilinx/Vitis_HLS/2024.1/include"
print(os.environ["XILINX_AP_INCLUDE"])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import seaborn as sns
import hls4ml
import math

(X_tr, y_tr), (X_te, y_te) = mnist.load_data()
X_tr = X_tr.reshape(-1, 784).astype('float32') / 255.0
X_te = X_te.reshape(-1, 784).astype('float32') / 255.0
X_train, X_val, y_train, y_val = train_test_split(
    X_tr, y_tr, test_size=0.1, random_state=42, stratify=y_tr
)
y_train_cat = to_categorical(y_train, 10)
y_val_cat   = to_categorical(y_val,   10)
y_test_cat  = to_categorical(y_te,    10)

print('Dataset loaded.')

## 2. Train Keras and QKeras models

We train two models with the same architecture:
- **Keras (FP32)**: standard float32 training (baseline reference)
- **QKeras (QAT)**: quantization-aware training with 8-bit weights (the model we'll deploy)

The QKeras model has already learned to be robust to fixed-point rounding,
so we expect it to show better agreement with the HLS C-sim output.

In [ ]:
# Keras FP32 baseline 
model_fp32 = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(784,)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax'),
], name='keras_fp32')
model_fp32.compile(optimizer='adam',
                   loss='sparse_categorical_crossentropy',
                   metrics=['accuracy'])
model_fp32.fit(X_train, y_train, epochs=10, batch_size=64,
               validation_data=(X_val, y_val), verbose=0)

_, acc_fp32 = model_fp32.evaluate(X_te, y_te, verbose=0)
print(f'Keras FP32 accuracy : {acc_fp32:.4f}')

# QKeras QAT model 
from qkeras import QDense, QActivation
from qkeras.quantizers import quantized_bits

def build_qkeras_model(wbits=8, ibits=0):
    return tf.keras.Sequential([
        QDense(64, input_shape=(784,),
               kernel_quantizer=quantized_bits(wbits, ibits),
               bias_quantizer=quantized_bits(wbits, ibits)),
        QActivation(f'quantized_relu({wbits},{ibits})'),
        QDense(32,
               kernel_quantizer=quantized_bits(wbits, ibits),
               bias_quantizer=quantized_bits(wbits, ibits)),
        QActivation(f'quantized_relu({wbits},{ibits})'),
        QDense(10,
               kernel_quantizer=quantized_bits(wbits, ibits),
               bias_quantizer=quantized_bits(wbits, ibits),
               activation='softmax'),
    ], name='qkeras_8b')

model_qk = build_qkeras_model(wbits=8, ibits=0)
model_qk.compile(optimizer='adam',
                 loss='categorical_crossentropy',
                 metrics=['accuracy'])
model_qk.fit(X_train, y_train_cat, epochs=20, batch_size=64,
             validation_data=(X_val, y_val_cat), verbose=0)

_, acc_qk = model_qk.evaluate(X_te, y_test_cat, verbose=0)
print(f'QKeras  8-bit accuracy : {acc_qk:.4f}')

## 3. Accumulator bit-width: choosing `ap_fixed<W,I>`

`ap_fixed<W,I>` in hls4ml applies to **accumulators and activations**, not only weights.

For a Dense layer with fan-in `N` and 8-bit weights:
- Each product uses 8 bits
- Summing `N` products can grow by up to `log₂(N)` extra bits
- Minimum integer bits needed: `ceil(log₂(N)) + 1` (the +1 is for the sign bit)

If `I` is too small → values saturate → accuracy drops dramatically.  
This is the most common cause of unexpected accuracy loss after conversion.

In [ ]:
layer_sizes = [784, 64, 32, 10]
weight_bits = 8

print('Accumulator bit-width analysis')
print('=' * 56)
print(f'{"Layer":<22} {"Fan-in":<10} {"Min I bits":<14} {"Recommended ap_fixed"}')
print('-' * 56)
for i in range(len(layer_sizes) - 1):
    fan_in  = layer_sizes[i]
    min_i   = math.ceil(math.log2(fan_in)) + 1
    rec_w   = weight_bits + min_i
    print(f'Layer {i+1} ({layer_sizes[i]:>3}→{layer_sizes[i+1]:<3})       '
          f'{fan_in:<10} {min_i:<14} ap_fixed<{rec_w},{min_i}>')

print()
print('Rule of thumb: I >= ceil(log2(fan_in)) + 1  to avoid overflow')

### Exercise 3.1 | Calculate accumulator bits for a new architecture

A new model has layers: **784 → 512 → 256 → 128 → 10** with 8-bit weights.
Calculate the minimum integer bits `I` needed for each layer's accumulator.

> **Hint:** `min_I = ceil(log2(fan_in)) + 1`. The +1 accounts for the sign bit.
> For overflow to occur, `I` must cover the full range of the accumulated sum.

**What to do:** Complete the `layer_sizes` list and run the analysis.
Then answer: *which layer is most at risk of overflow if you use `ap_fixed<16,6>`?*

In [ ]:
# Exercise 3.1
# TODO: update layer_sizes with the new architecture
layer_sizes = [784, ???, ???, ???, 10]  # fill in the hidden layers
weight_bits = 8

# Hint: the loop below is already correct — just fix layer_sizes above
print('Accumulator bit-width analysis')
print('=' * 56)
print(f'{"Layer":<22} {"Fan-in":<10} {"Min I bits":<14} {"Recommended ap_fixed"}')
print('-' * 56)
for i in range(len(layer_sizes) - 1):
    fan_in  = layer_sizes[i]
    min_i   = math.ceil(math.log2(fan_in)) + 1
    rec_w   = weight_bits + min_i
    print(f'Layer {i+1} ({layer_sizes[i]:>3}→{layer_sizes[i+1]:<3})       '
          f'{fan_in:<10} {min_i:<14} ap_fixed<{rec_w},{min_i}>')

# TODO: which layer needs the most integer bits? Why?
# YOUR ANSWER (as a comment):
# The most critical layer is ...

> **Bonus:** What happens if you use `ap_fixed<8,0>` for the first layer? Would it overflow? By how much?
> Try to calculate it analytically before running the next section.

## 4. Convert to HLS and run C-simulation

`hls_model.compile()` invokes Vivado HLS in **C-simulation mode**:
- Runs in seconds (no synthesis, no place-and-route)
- Uses the **exact same fixed-point arithmetic** that will be synthesized
- This is not an approximation, it is the hardware model

Adjust `Precision` and `ReuseFactor` below and re-run to explore the tradeoff.

In [ ]:
# Parameters to adjust
AP_FIXED    = 'ap_fixed<16,6>'   # try: <8,4>, <8,0>, <16,6>, <4,2>
REUSE_FACTOR = 4                  # try: 1, 4, 8, 32
# 

def make_hls_model(keras_model, precision, reuse, output_dir):
    config = hls4ml.utils.config_from_keras_model(
        keras_model, granularity='name'
    )
    config['Model']['Precision']    = precision
    config['Model']['ReuseFactor']  = reuse
    hls_m = hls4ml.converters.convert_from_keras_model(
        keras_model,
        hls_config=config,
        output_dir=output_dir,
        backend='Vitis'
    )
    hls_m.compile()
    return hls_m

print(f'Converting QKeras model  →  {AP_FIXED}  RF={REUSE_FACTOR}')
hls_qk = make_hls_model(model_qk,   AP_FIXED, REUSE_FACTOR, 'hls_qkeras')

print(f'Converting Keras FP32    →  {AP_FIXED}  RF={REUSE_FACTOR}')
hls_fp32 = make_hls_model(model_fp32, AP_FIXED, REUSE_FACTOR, 'hls_fp32')


### Exercise 4.1 | Explore precision vs accuracy tradeoff

Run the conversion with **4 different configurations** and record the C-sim accuracy.
Fill in the table once you have the results.

| AP_FIXED | ReuseFactor | QKeras C-sim acc | FP32 C-sim acc | Notes |
|---|---|---|---|---|
| `ap_fixed<16,6>` | 4 | | | baseline |
| `ap_fixed<8,4>` | 4 | | | fewer bits |
| `ap_fixed<8,0>` | 4 | | | danger zone |
| `ap_fixed<16,6>` | 32 | | | high reuse |




> **Hint:** Change `AP_FIXED` and `REUSE_FACTOR` in the cell below and re-run section 4.
> Pay special attention to `ap_fixed<8,0>` — what does `I=0` mean for overflow?

**Question:** What is the minimum bit-width that keeps accuracy above 95%?

## 5. Confusion matrix: Keras / QKeras vs HLS C-sim

Side-by-side comparison for both models.  
The confusion matrix from HLS C-sim is the same one you would see after full synthesis.

In [ ]:
# Predictions 
y_fp32_k   = np.argmax(model_fp32.predict(X_te, verbose=0), axis=1)
y_fp32_hls = np.argmax(hls_fp32.predict(X_te),              axis=1)

y_qk_k     = np.argmax(model_qk.predict(X_te, verbose=0),   axis=1)
y_qk_hls   = np.argmax(hls_qk.predict(X_te),                axis=1)

acc = {
    'fp32_k':   np.mean(y_fp32_k   == y_te),
    'fp32_hls': np.mean(y_fp32_hls == y_te),
    'qk_k':     np.mean(y_qk_k     == y_te),
    'qk_hls':   np.mean(y_qk_hls   == y_te),
}
agree = {
    'fp32':  np.mean(y_fp32_k == y_fp32_hls),
    'qkeras': np.mean(y_qk_k  == y_qk_hls),
}

# Summary table 
print(f'Precision: {AP_FIXED}  ·  ReuseFactor: {REUSE_FACTOR}')
print('=' * 60)
print(f'{"":<28} {"Keras FP32":<16} {"QKeras (QAT)"}')
print('-' * 60)
print(f'{"Keras accuracy":<28} {acc["fp32_k"]:.4f}          {acc["qk_k"]:.4f}')
print(f'{"HLS C-sim accuracy":<28} {acc["fp32_hls"]:.4f}          {acc["qk_hls"]:.4f}')
print(f'{"Keras ↔ HLS agreement":<28} {agree["fp32"]:.4f}          {agree["qkeras"]:.4f}')
print(f'{"Discrepancies":<28} {int((1-agree["fp32"])*10000):<16} {int((1-agree["qkeras"])*10000)}')

# Confusion matrices (2×2 grid) 
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
pairs = [
    (y_fp32_k,   'Keras FP32',          'Purples', axes[0,0]),
    (y_fp32_hls, f'HLS C-sim ({AP_FIXED})', 'Blues',   axes[0,1]),
    (y_qk_k,     'QKeras (QAT)',         'Purples', axes[1,0]),
    (y_qk_hls,   f'HLS C-sim ({AP_FIXED})', 'Greens',  axes[1,1]),
]
for y_pred, title, cmap, ax in pairs:
    cm = confusion_matrix(y_te, y_pred)
    acc_v = np.mean(y_pred == y_te)
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, ax=ax, cbar=False)
    ax.set_title(f'{title}\nacc={acc_v:.4f}', fontsize=11)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')

fig.suptitle(f'Confusion matrices  ·  {AP_FIXED}  RF={REUSE_FACTOR}', fontsize=13)
plt.tight_layout()
plt.show()

### Exercise 5.1 | Compare confusion matrices

After running section 5 with at least two different precisions, answer:

1. Which digit class shows the most disagreements between Keras and HLS?
2. Does QKeras show fewer discrepancies than FP32? Why do you think that is?
3. What does a high Keras↔HLS agreement (e.g. 99.8%) tell you about the quantization?

> **Hint:** Look at the off-diagonal elements of the confusion matrix, those are the misclassifications. Cross-reference with section 6 disagreement cases.

**Your answers (edit this cell):**
1. ...
2. ...
3. ...

## 6. Disagreement cases: Keras/QKeras vs HLS C-sim

These are the samples where the software model and the hardware model **disagree**.
Inspecting them helps diagnose *why* the discrepancy happens:
- Ambiguous samples → likely rounding at a decision boundary (benign)
- Clear samples that flip → possible overflow or activation approximation issue

In [ ]:
def show_disagreements(y_soft, y_hls, y_true, X, label_soft, label_hls, n_show=8):
    disagree_idx = np.where(y_soft != y_hls)[0]
    print(f'{label_soft} vs {label_hls}: {len(disagree_idx)} disagreements '
          f'({len(disagree_idx)/len(y_true)*100:.2f}%)')

    if len(disagree_idx) == 0:
        print('  Perfect agreement — no cases to show.')
        return

    n = min(n_show, len(disagree_idx))
    fig, axes = plt.subplots(1, n, figsize=(n * 1.6, 2.4))
    if n == 1:
        axes = [axes]
    for ax, idx in zip(axes, disagree_idx[:n]):
        ax.imshow(X[idx].reshape(28, 28), cmap='gray')
        ax.set_title(
            f'True: {y_true[idx]}\n'
            f'{label_soft}: {y_soft[idx]}\n'
            f'{label_hls}: {y_hls[idx]}',
            fontsize=7
        )
        ax.axis('off')
    plt.suptitle(f'Disagreement cases — {label_soft} vs {label_hls}', fontsize=10)
    plt.tight_layout()
    plt.show()

# Keras FP32 vs its HLS model
show_disagreements(y_fp32_k, y_fp32_hls, y_te, X_te,
                   label_soft='Keras', label_hls='HLS')

print()

# QKeras vs its HLS model
show_disagreements(y_qk_k, y_qk_hls, y_te, X_te,
                   label_soft='QKeras', label_hls='HLS')

### Exercise 6.1 | Diagnose a disagreement case

Pick **one** disagreement image from section 6 where a clear digit was misclassified.
Hypothesize whether the cause is:
- **(a) Rounding at a decision boundary**: the sample is ambiguous
- **(b) Overflow**: the integer bits `I` in `ap_fixed<W,I>` are too small
- **(c) Activation approximation**: hls4ml uses a lookup table for ReLU/softmax

> **Hint:** If the same digit is consistently misclassified across different precisions,
> it's likely (a). If it only fails with low `I`, suspect (b).
> Re-run with `ap_fixed<8,0>` to force overflow and compare the disagreement count.

**Your analysis (edit this cell):**
- Sample index: ...
- True label: ... | Keras prediction: ... | HLS prediction: ...
- Likely cause: ...
- Evidence: ...

## 7. Resource utilization: synthesis report

hls_model.build(synth=True) runs Vivado HLS synthesis (5–15 min).
The report gives LUT, FF, BRAM, DSP estimates for the xczu3eg-sfvc784-1-e.
Note: LUT counts from HLS synthesis are typically overestimated 
vs post-implementation (keep that in mind when interpreting the numbers).

In [ ]:

# Parameters 
PART        = 'xczu3eg-sfvc784-1-e'
CLOCK_NS    = 10          # 100 MHz target
SYNTH_DIR   = 'hls_synth_qkeras'

# xczu3eg-sfvc784-1-e total resources (Zynq UltraScale+)
TOTAL = {
    'LUT':  70560,
    'FF':   141120,
    'BRAM': 216,
    'DSP':  360,
}

# Convert with target part
config_synth = hls4ml.utils.config_from_keras_model(model_qk, granularity='name')
config_synth['Model']['Precision']   = AP_FIXED
config_synth['Model']['ReuseFactor'] = REUSE_FACTOR

hls_synth = hls4ml.converters.convert_from_keras_model(
    model_qk,
    hls_config=config_synth,
    output_dir=SYNTH_DIR,
    backend='Vitis',
    part=PART,
    clock_period=CLOCK_NS,
)

# Run synthesis (grab a coffee) 
print(f'Precision: {AP_FIXED}  ·  ReuseFactor: {REUSE_FACTOR}  ·  Clock: {CLOCK_NS} ns')
hls_synth.build(csim=False, synth=True, export=False)
print('Synthesis complete.')

# Parse and print report 
report = hls4ml.report.read_vivado_report(SYNTH_DIR)
hls4ml.report.print_vivado_report(report)

# Resource utilization table with % of xczu3eg 
try:
    synth = report['CSynthesisReport']
    used = {
        'LUT':  int(synth.get('LUT',  synth.get('LUTS', 0))),
        'FF':   int(synth.get('FF',   0)),
        'BRAM': int(synth.get('BRAM', synth.get('BRAM_18K', 0))),
        'DSP':  int(synth.get('DSP',  synth.get('DSP48E', 0))),
    }

    print(f'\nResource utilization — {PART}')
    print(f'Precision: {AP_FIXED}  ·  RF={REUSE_FACTOR}  ·  Clock: {CLOCK_NS} ns')
    print('=' * 52)
    print(f'{"Resource":<10} {"Used":>8} {"Available":>12} {"Utilization":>14}')
    print('-' * 52)
    for res, u in used.items():
        total = TOTAL[res]
        pct   = u / total * 100
        bar   = '#' * int(pct / 5)  # one # per 5%
        print(f'{res:<10} {u:>8,} {total:>12,} {pct:>12.1f}%  {bar}')
    print('=' * 52)

    latency_min = report['CSynthesisReport'].get('LatencyMin', 'N/A')
    latency_max = report['CSynthesisReport'].get('LatencyMax', 'N/A')
    print(f'Latency: {latency_min} – {latency_max} cycles  '
          f'({int(latency_max) * CLOCK_NS / 1000:.3f} µs @ {1000//CLOCK_NS} MHz)'
          if latency_max != 'N/A' else f'Latency: {latency_min} – {latency_max} cycles')

except Exception as e:
    print(f'Could not parse report automatically: {e}')
    print('Check the raw report above for resource numbers.')

### Exercise 7.1 | Resource budget analysis

The target device is `xczu3eg-sfvc784-1-e` with these total resources:

| Resource | Available |
|---|---|
| LUT | 70,560 |
| FF | 141,120 |
| BRAM | 216 |
| DSP | 360 |

After running synthesis, answer:

1. Which resource is the bottleneck (closest to 100% utilization)?
2. If you increase `ReuseFactor` from 4 to 32, what do you expect to happen to LUT usage? And to latency?
3. What `ReuseFactor` would you choose to keep LUT utilization below 50%?

> **Hint:** Higher `ReuseFactor` = more time-multiplexing of operations = fewer parallel units = fewer LUTs,
> but more clock cycles needed (higher latency). It's a classic area-vs-latency tradeoff.

**Your answers (edit this cell):**
1. Bottleneck resource: ...
2. Effect of RF=32: LUTs → ... | Latency → ...
3. Recommended ReuseFactor: ...